ET-SSL Pre-Training Notebook
**Anomaly Detection in Encrypted Network Traffic using Self-Supervised Contrastive Learning**

Paper: s41598-025-08568-0 (Scientific Reports 2025)

Setup
Runtime: GPU (T4 or A100 recommended)
Estimated training time: ~45 min on T4 per dataset


## 1. Environment Setup — Clone Repo & Install Dependencies

In [ ]:
# ── 1a. Clone the Sentinel repository ────────────────────────────────────────
import os, sys

REPO_URL  = "https://github.com/muro906/Sentinel.git"   # ← update this
REPO_DIR  = "Sentinel"
HYBRID_DIR = f"{REPO_DIR}/hybrid-detection"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL}")
else:
    os.system(f"git -C {REPO_DIR} pull --ff-only")   # keep up to date

# Add hybrid-detection to Python path so all config/model imports resolve
if HYBRID_DIR not in sys.path:
    sys.path.insert(0, HYBRID_DIR)

print("Repo ready. sys.path:", sys.path[:3])

In [ ]:
# ── Kaggle API Setup ──────────────────────────────────────────────────────────
import os
import shutil
import subprocess

# Copy kaggle.json from Drive to ~/.kaggle/
KAGGLE_KEY_PATH = '/content/drive/MyDrive/kaggle.json'  # Adjust path if different
KAGGLE_DIR = os.path.expanduser('~/.kaggle')

os.makedirs(KAGGLE_DIR, exist_ok=True)
shutil.copy(KAGGLE_KEY_PATH, f'{KAGGLE_DIR}/kaggle.json')
os.chmod(f'{KAGGLE_DIR}/kaggle.json', 0o600)

# Install Kaggle API
subprocess.run(['pip', 'install', '-q', 'kaggle'], check=True)

print('Kaggle API configured')

In [ ]:
# ── 1b. Install dependencies ──────────────────────────────────────────────────
# (Colab already has torch/sklearn; optuna and joblib may need installing)
import subprocess
subprocess.run(["pip", "install", "-q", "optuna", "joblib", "tqdm"], check=True)
print("Dependencies installed.")

## 2. Imports from Sentinel Repo

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# ── Config ────────────────────────────────────────────────────────────────────
from config.constants import (
    FEATURE_DIM,
    CONTINUOUS_INDICES, CATEGORICAL_INDICES,
    BATCH_SIZE, LEARNING_RATE, LR_DECAY_FACTOR, LR_DECAY_EPOCHS,
    NUM_EPOCHS, TEMPERATURE_TAU, GAMMA, ALPHA_EMA,
    TRAIN_RATIO, VAL_RATIO, TEST_RATIO,
    EMBEDDING_DIM, PROJECTION_DIM,
)
from config.column_maps import (
    CIC_DARKNET2020_COLUMN_MAP,
    CIC_IDS2018_COLUMN_MAP,
    CICFLOWMETER_PROTO_MAP,
    UNSW_MAP,
    LABEL_NORMAL,
)

# ── Model components ──────────────────────────────────────────────────────────
from model.et_ssl      import ETSSLModel
from model.loss        import NTXentLoss
from model.preprocessor import Augmenter, TrafficScaler
from model.dataset     import TrafficDataset

# ── Feature builder (same fn used by the detection service) ──────────────────
from feature_extractor.feature_builder import build_feature_matrix

print(f"Feature dim: {FEATURE_DIM}")
print(f"Continuous indices: {CONTINUOUS_INDICES}")
print(f"Categorical indices: {CATEGORICAL_INDICES}")
print("All imports OK.")

## 3. Configuration

In [ ]:
DATASETS = ['darknet', 'ids2018', 'unsw']
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Paper hyperparameters — imported from constants, used as search defaults
DEFAULT_CONFIG = dict(
    lr          = LEARNING_RATE,
    batch_size  = BATCH_SIZE,
    epochs      = NUM_EPOCHS,
    temperature = TEMPERATURE_TAU,
    gamma       = GAMMA,
    alpha_ema   = ALPHA_EMA,
    dropout     = 0.3,
    lr_decay    = LR_DECAY_FACTOR,
    lr_decay_ep = LR_DECAY_EPOCHS,
    embed_dim   = EMBEDDING_DIM,
    proj_dim    = PROJECTION_DIM,
    hidden_dims    = (128, 256, 128),
    n_hidden       = 3,
    hidden_w1      = 128,
    hidden_w2      = 256,
    hidden_w3      = 128,
    weight_decay   = 0.0,
    noise_std      = 0.05,
    dropout_max_feat = 3,
    jitter_lo      = 0.9,
    jitter_hi      = 1.1,
)

# Mount Google Drive for dataset + artifact storage
from google.colab import drive
drive.mount('/content/drive')
DATASET_DIR = '/content/drive/MyDrive/Sentinel/datasets'
EXPORT_DIR  = '/content/drive/MyDrive/Sentinel/checkpoints'
import os; os.makedirs(EXPORT_DIR, exist_ok=True)
print(f"Dataset dir : {DATASET_DIR}")
print(f"Export  dir : {EXPORT_DIR}")

## 4. Data Loading & Preprocessing

In [ ]:
# ── Download & Merge All Datasets from Kaggle ───────────────────────────────
import subprocess
import glob

KAGGLE_DATASETS = {
    'darknet': 'peterfriedrich1/cicdarknet2020-internet-traffic',
    'ids2018': 'solarmainframe/ids-intrusion-csv',
    'unsw':    'mrwellsdavid/unsw-nb15',
}

# Output file names expected by the training cells
DATASET_OUTPUT_FILES = {
    'darknet': f'{DATASET_DIR}/darknet2020.csv',
    'ids2018': f'{DATASET_DIR}/cic_ids2018.csv',
    'unsw':    f'{DATASET_DIR}/unsw_nb15.csv',
}

def download_and_merge(dataset_name, kaggle_path, output_path, sample_limit=None, balance=True):
    """
    Download a single dataset from Kaggle, merge all its CSV files,
    standardize columns, optionally balance classes, and save to output_path.
    CSVs from different datasets are NEVER mixed.
    """
    import tempfile

    DATASET_CONFIGS = {
        'darknet': {
            'col_map':      CIC_DARKNET2020_COLUMN_MAP,
            'label_normal': LABEL_NORMAL['darknet'],
            'fix_proto':    True,
        },
        'ids2018': {
            'col_map':      CIC_IDS2018_COLUMN_MAP,
            'label_normal': LABEL_NORMAL['ids2018'],
            'fix_proto':    True,
        },
        'unsw': {
            'col_map':      UNSW_MAP,
            'label_normal': LABEL_NORMAL['unsw'],
            'fix_proto':    False,
        },
    }
    cfg = DATASET_CONFIGS[dataset_name]

    with tempfile.TemporaryDirectory() as tmpdir:
        print(f"\nDownloading {dataset_name} from Kaggle...")
        subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', kaggle_path, '-p', tmpdir, '--unzip'],
            check=True
        )

        # Find ALL CSV files belonging to this single dataset
        # Search both lowercase and uppercase extensions (Kaggle datasets vary)
        csv_files = sorted(
            glob.glob(f'{tmpdir}/**/*.csv', recursive=True) +
            glob.glob(f'{tmpdir}/**/*.CSV', recursive=True)
        )
        # For UNSW-NB15: keep only the 4 raw data files (UNSW-NB15_1..4.csv).
        # Exclude UNSW-NB15_LIST_EVENTS.csv (event metadata, no traffic features),
        # UNSW_NB15_training-set.csv and UNSW_NB15_testing-set.csv (pre-split
        # subsets with a different, reduced column schema — mixing them with the
        # raw files causes shape mismatches and duplicate rows).
        if dataset_name == 'unsw':
            import re
            csv_files = [f for f in csv_files
                         if re.match(r'UNSW-NB15_[1-4]\.csv$', os.path.basename(f), re.IGNORECASE)]
            print(f"  UNSW filter applied — keeping: {[os.path.basename(f) for f in csv_files]}")
        print(f"  Found {len(csv_files)} CSV file(s): {[os.path.basename(f) for f in csv_files]}")
        if not csv_files:
            # Print full directory tree to help diagnose the actual file names/paths
            print('  Directory contents after unzip:')
            for root, dirs, files in os.walk(tmpdir):
                level = root.replace(tmpdir, '').count(os.sep)
                print(f'  {"  " * level}{os.path.basename(root)}/')
                for fname in files:
                    print(f'  {"  " * (level+1)}{fname}')
            raise FileNotFoundError(
                f'No CSV files found for {dataset_name}. '
                f'Check the directory tree above for the actual file names.'
            )

        # Read CSVs in chunks — standardise, label, and sample per chunk so the
        # full dataset never lives in RAM at once. Critical for large datasets
        # like IDS2018 (~6 GB) which crash Colab if loaded entirely into memory.
        CHUNK_SIZE = 100_000   # rows per chunk; tune down to 50_000 if still OOMing
        normal_chunks  = []    # sampled normal rows collected across all chunks/files
        anomaly_chunks = []    # sampled anomaly rows collected across all chunks/files
        # Per-class row budget: collect up to 2× sample_limit so the final balance
        # step has enough rows to draw from after deduplication.
        per_class_budget = (sample_limit * 2) if sample_limit else None

        for f in csv_files:
            print(f"  Reading (chunked): {os.path.basename(f)}")
            try:
                reader = pd.read_csv(f, low_memory=False, on_bad_lines='skip',
                                     chunksize=CHUNK_SIZE)
            except TypeError:
                reader = pd.read_csv(f, low_memory=False, error_bad_lines=False,
                                     chunksize=CHUNK_SIZE)

            for chunk in reader:
                # Standardise columns
                chunk = chunk.rename(columns=cfg['col_map']).fillna(0)

                # Fix numeric protocol for CIC datasets (6→tcp, 17→udp)
                if cfg['fix_proto'] and 'proto' in chunk.columns:
                    chunk['proto'] = chunk['proto'].map(CICFLOWMETER_PROTO_MAP).fillna('other')

                # Binary label
                chunk['label_binary'] = (chunk['label'] != cfg['label_normal']).astype(int)

                # Split and sample per chunk to limit memory accumulation
                norm  = chunk[chunk['label_binary'] == 0]
                anom  = chunk[chunk['label_binary'] == 1]

                if per_class_budget is not None:
                    # Only keep what we still need
                    need_norm = max(0, per_class_budget - sum(len(c) for c in normal_chunks))
                    need_anom = max(0, per_class_budget - sum(len(c) for c in anomaly_chunks))
                    if need_norm > 0:
                        normal_chunks.append(norm.iloc[:need_norm])
                    if need_anom > 0:
                        anomaly_chunks.append(anom.iloc[:need_anom])
                    # Stop reading this file early if both budgets are filled
                    if (sum(len(c) for c in normal_chunks) >= per_class_budget and
                            sum(len(c) for c in anomaly_chunks) >= per_class_budget):
                        print(f"    Budget reached — stopping early")
                        break
                else:
                    normal_chunks.append(norm)
                    anomaly_chunks.append(anom)

        normal_df  = pd.concat(normal_chunks,  ignore_index=True) if normal_chunks  else pd.DataFrame()
        anomaly_df = pd.concat(anomaly_chunks, ignore_index=True) if anomaly_chunks else pd.DataFrame()
        print(f"  Collected — normal: {len(normal_df):,}  anomaly: {len(anomaly_df):,}")

        # Balance classes
        if balance:
            if sample_limit:
                n = min(len(normal_df), len(anomaly_df), sample_limit)
            else:
                n = min(len(normal_df), len(anomaly_df))
            normal_df  = normal_df.sample(n,  random_state=42)
            anomaly_df = anomaly_df.sample(n, random_state=42)
            print(f"  Balanced to {n:,} samples per class  ({2*n:,} total)")
        elif sample_limit:
            total = sample_limit * 2
            frac_norm = len(normal_df) / max(len(normal_df) + len(anomaly_df), 1)
            normal_df  = normal_df.sample(min(len(normal_df),  int(total * frac_norm)),      random_state=42)
            anomaly_df = anomaly_df.sample(min(len(anomaly_df), total - len(normal_df)), random_state=42)

        combined = pd.concat([normal_df, anomaly_df], ignore_index=True).sample(
            frac=1, random_state=42).reset_index(drop=True)
        print(f"  Final combined shape: {combined.shape}")
        print(f"  Label distribution: {combined['label_binary'].value_counts().to_dict()}")

        del normal_chunks, anomaly_chunks, normal_df, anomaly_df  # free RAM immediately

        combined.to_csv(output_path, index=False)
        print(f"  Saved → {output_path}  (shape: {combined.shape})")

    return output_path

# Download each dataset separately — no cross-dataset merging
os.makedirs(DATASET_DIR, exist_ok=True)
DATASET_FILES = {}
for ds in DATASETS:
    # Skip if already downloaded
    if os.path.exists(DATASET_OUTPUT_FILES[ds]):
        print(f"{ds}: already exists at {DATASET_OUTPUT_FILES[ds]}, skipping download.")
        DATASET_FILES[ds] = DATASET_OUTPUT_FILES[ds]
    else:
        DATASET_FILES[ds] = download_and_merge(
            ds,
            KAGGLE_DATASETS[ds],
            DATASET_OUTPUT_FILES[ds],
            sample_limit=50000,  # max samples per class; set None to use all
            balance=True
        )

print("\nAll datasets ready:")
for ds, path in DATASET_FILES.items():
    print(f"  {ds}: {path}")

## 5. Train / Val / Test Split & Scaling

In [ ]:
from sklearn.model_selection import train_test_split

# These dicts will hold per-dataset splits and scalers for the training loop
DATASET_SPLITS  = {}  # ds -> {X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test}
DATASET_SCALERS = {}  # ds -> fitted TrafficScaler

for DATASET_CHOICE in DATASETS:
    print(f"\n{'='*60}")
    print(f"Preparing splits for: {DATASET_CHOICE}")

    df = pd.read_csv(DATASET_FILES[DATASET_CHOICE], low_memory=False)

    # Build feature matrix
    records = df.to_dict(orient='records')
    X_all   = build_feature_matrix(records).astype(np.float32)
    y_all   = df['label_binary'].to_numpy() if 'label_binary' in df.columns else (df['label'] != 0).astype(int).to_numpy()
    if len(y_all) != len(X_all):
        y_all = y_all[:len(X_all)]

    val_frac_of_trainval = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X_all, y_all, test_size=TEST_RATIO, stratify=y_all, random_state=42
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=val_frac_of_trainval,
        stratify=y_trainval, random_state=42
    )
    print(f"  Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

    scaler         = TrafficScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)
    X_test_scaled  = scaler.transform(X_test)

    assert np.array_equal(X_train_scaled[:, 12:], X_train[:, 12:]), "One-hot columns were modified!"

    DATASET_SPLITS[DATASET_CHOICE] = {
        'X_train': X_train_scaled,
        'X_val':   X_val_scaled,
        'X_test':  X_test_scaled,
        'y_train': y_train,
        'y_val':   y_val,
        'y_test':  y_test,
    }
    DATASET_SCALERS[DATASET_CHOICE] = scaler

print("\nAll splits ready:", list(DATASET_SPLITS.keys()))

## 6. Training Utilities

In [ ]:
# All classes (Augmenter, TrafficDataset, ETSSLModel, NTXentLoss) are already
# imported from the repo in Cell 2. This cell only defines the training loop.

@torch.no_grad()
def get_embeddings(model, X_np, device, batch_size=512):
    model.eval()
    dl = DataLoader(torch.from_numpy(X_np).float(), batch_size=batch_size)
    return torch.cat([model.encode(b.to(device)) for b in dl]).cpu().numpy()

def compute_auc(embeddings, centroid, labels):
    from sklearn.metrics import roc_auc_score
    scores = ((embeddings - centroid[None, :]) ** 2).sum(axis=1)
    try:    return roc_auc_score(labels, scores)
    except: return 0.5

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total = 0.0
    for batch in loader:
        x_orig, x_aug = batch[0].to(device), batch[1].to(device)
        optimizer.zero_grad()
        _, h     = model(x_orig)
        _, h_aug = model(x_aug)
        loss = loss_fn(h, h_aug)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def eval_epoch(model, loader, loss_fn, device):
    model.eval()
    total = 0.0
    for batch in loader:
        x_orig, x_aug = batch[0].to(device), batch[1].to(device)
        _, h     = model(x_orig)
        _, h_aug = model(x_aug)
        total   += loss_fn(h, h_aug).item()
    return total / len(loader)

def run_training(config, X_train_s, X_val_s, y_val, device, trial_name='default'):
    aug    = Augmenter(
        noise_std=config.get('noise_std', 0.05),
        dropout_max=config.get('dropout_max_feat', 3),
        jitter_range=(config.get('jitter_lo', 0.9), config.get('jitter_hi', 1.1)),
    )
    # Train loader: TrafficDataset produces (x_orig, x_aug) pairs via augmentation.
    # Augmentation is intentional here — contrastive learning requires two views
    # of each training sample.
    ds     = TrafficDataset(X_train_s, augmenter=aug)
    dl     = DataLoader(ds, batch_size=config['batch_size'], shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=True)
    # Val loader: plain TensorDataset — no augmentation.
    # Validation measures the model on clean, unmodified data.
    # Using augmented val samples would give a noisy, unreliable loss estimate
    # and an artificially inflated AUC (augmented pairs are "easier" for the model).
    val_t  = torch.from_numpy(X_val_s).float()
    val_ds = torch.utils.data.TensorDataset(val_t, val_t)  # (x_orig, x_orig) — no augmentation
    val_dl = DataLoader(val_ds, batch_size=512, shuffle=False, num_workers=2)

    model     = ETSSLModel(
                    hidden_dims=config.get('hidden_dims', (128, 256, 128)),
                    embed_dim=config['embed_dim'],
                    proj_dim=config['proj_dim'],
                    dropout=config['dropout'],
                ).to(device)
    # AdamW: Adam + decoupled weight decay (better for contrastive learning)
    # weight_decay=0.0 falls back to plain Adam
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config.get('weight_decay', 0.0),
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=config['lr_decay_ep'], gamma=config['lr_decay'])
    loss_fn   = NTXentLoss(temperature=config['temperature'])

    best_auc, best_state, best_centroid = 0.0, None, None
    history, last_centroid = [], None

    for epoch in range(1, config['epochs'] + 1):
        train_loss = train_one_epoch(model, dl, optimizer, loss_fn, device)
        val_loss   = eval_epoch(model, val_dl, loss_fn, device)
        scheduler.step()

        if epoch % 5 == 0 or epoch == 1:
            z_train  = get_embeddings(model, X_train_s, device)
            centroid = z_train.mean(axis=0)
            if last_centroid is not None:
                centroid = (config['alpha_ema'] * last_centroid
                            + (1 - config['alpha_ema']) * centroid)
            last_centroid = centroid

            z_val = get_embeddings(model, X_val_s, device)
            auc   = compute_auc(z_val, centroid, y_val)
            history.append({'epoch': epoch, 'train_loss': train_loss,
                            'val_loss': val_loss, 'val_auc': auc})
            print(f"[{trial_name}] Ep {epoch:3d} | "
                  f"train={train_loss:.4f} val={val_loss:.4f} AUC={auc:.4f}")
            if auc > best_auc:
                best_auc      = auc
                best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                best_centroid = centroid.copy()
        else:
            history.append({'epoch': epoch, 'train_loss': train_loss,
                            'val_loss': val_loss})

    model.load_state_dict(best_state)
    return model, best_centroid, best_auc, history

9. Baseline + Optuna Search per Dataset


In [ ]:
import optuna
import json as _json
import joblib as _joblib
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Per-dataset training results ──────────────────────────────────────────────
TRAINED_MODELS = {}   # ds -> {model, centroid, scaler, threshold, meta}

for DATASET_CHOICE in DATASETS:
    print(f"\n{'#'*60}")
    print(f"# TRAINING ON: {DATASET_CHOICE}")
    print(f"{'#'*60}")

    splits = DATASET_SPLITS[DATASET_CHOICE]
    X_train_scaled = splits['X_train']
    X_val_scaled   = splits['X_val']
    X_test_scaled  = splits['X_test']
    y_train        = splits['y_train']
    y_val          = splits['y_val']
    y_test         = splits['y_test']
    scaler         = DATASET_SCALERS[DATASET_CHOICE]

    PLOTS_DIR = f'{EXPORT_DIR}/plots/{DATASET_CHOICE}'
    os.makedirs(PLOTS_DIR, exist_ok=True)

    # ── Baseline ──────────────────────────────────────────────────────────────
    print("\nStep 1: Baseline training with paper hyperparameters...")
    baseline_model, baseline_centroid, baseline_auc, baseline_history = run_training(
        DEFAULT_CONFIG, X_train_scaled, X_val_scaled, y_val, DEVICE, trial_name='baseline'
    )
    print(f"Baseline val AUC: {baseline_auc:.4f}")

    # ── Optuna hyperparameter search ──────────────────────────────────────────
    print("\nStep 2: Optuna hyperparameter search...")

    def objective(trial):
        config = dict(
            lr          = trial.suggest_float('lr', 1e-4, 3e-3, log=True),
            batch_size  = trial.suggest_categorical('batch_size', [128, 256, 512]),
            epochs      = 60,
            temperature = trial.suggest_categorical('temperature', [0.05, 0.1, 0.2, 0.5]),
            gamma       = trial.suggest_float('gamma', 0.1, 1.0),
            alpha_ema   = trial.suggest_categorical('alpha_ema', [0.9, 0.95, 0.99]),
            dropout     = trial.suggest_float('dropout', 0.1, 0.5),
            lr_decay    = trial.suggest_categorical('lr_decay', [0.9, 0.95, 0.98]),
            lr_decay_ep = trial.suggest_categorical('lr_decay_ep', [5, 10, 20]),
            n_hidden    = trial.suggest_categorical('n_hidden', [2, 3]),
            hidden_w1   = trial.suggest_categorical('hidden_w1', [64, 128, 256]),
            hidden_w2   = trial.suggest_categorical('hidden_w2', [128, 256, 512]),
            hidden_w3   = trial.suggest_categorical('hidden_w3', [64, 128, 256]),
            embed_dim   = trial.suggest_categorical('embed_dim', [32, 64, 128]),
            proj_dim    = trial.suggest_categorical('proj_dim',  [16, 32, 64]),
            noise_std        = trial.suggest_float('noise_std', 0.01, 0.2, log=True),
            dropout_max_feat = trial.suggest_int('dropout_max_feat', 1, 5),
            jitter_lo        = trial.suggest_float('jitter_lo', 0.80, 0.95),
            jitter_hi        = trial.suggest_float('jitter_hi', 1.05, 1.30),
            weight_decay     = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True),
        )
        if config['n_hidden'] == 2:
            config['hidden_dims'] = (config['hidden_w1'], config['hidden_w2'])
        else:
            config['hidden_dims'] = (config['hidden_w1'], config['hidden_w2'], config['hidden_w3'])
        _, _, auc, _ = run_training(config, X_train_scaled, X_val_scaled, y_val,
                                    DEVICE, trial_name=f"trial-{trial.number}")
        return auc

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.enqueue_trial({**DEFAULT_CONFIG, 'n_hidden': 3, 'hidden_w1': 128,
                         'hidden_w2': 256, 'hidden_w3': 128,
                         'noise_std': 0.05, 'dropout_max_feat': 3,
                         'jitter_lo': 0.9, 'jitter_hi': 1.1})
    study.optimize(objective, n_trials=50, show_progress_bar=True)
    print(f"Best Optuna AUC: {study.best_value:.4f} | Params: {study.best_params}")

    # ── Final training ─────────────────────────────────────────────────────────
    print("\nStep 3: Final training with best hyperparameters (150 epochs)...")
    best_config = {**DEFAULT_CONFIG, **study.best_params}
    best_config['epochs'] = 150
    if best_config.get('n_hidden', 3) == 2:
        best_config['hidden_dims'] = (best_config['hidden_w1'], best_config['hidden_w2'])
    else:
        best_config['hidden_dims'] = (best_config['hidden_w1'], best_config['hidden_w2'], best_config['hidden_w3'])

    final_model, final_centroid, final_auc, final_history = run_training(
        best_config, X_train_scaled, X_val_scaled, y_val, DEVICE, trial_name='final'
    )
    print(f"Final val AUC: {final_auc:.4f}")

    # ── Threshold calibration ──────────────────────────────────────────────────
    from sklearn.metrics import roc_auc_score, f1_score, classification_report
    z_val      = get_embeddings(final_model, X_val_scaled, DEVICE)
    val_scores = ((z_val - final_centroid[None, :]) ** 2).sum(axis=1)
    thresholds = np.linspace(val_scores.min(), val_scores.max(), 300)
    best_f1, best_thresh = 0.0, thresholds[0]
    for t in thresholds:
        preds = (val_scores > t).astype(int)
        f1 = f1_score(y_val, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, t
    print(f"Calibrated threshold: {best_thresh:.6f}  (val F1: {best_f1:.4f})")

    # ── Test evaluation ────────────────────────────────────────────────────────
    z_test      = get_embeddings(final_model, X_test_scaled, DEVICE)
    test_scores = ((z_test - final_centroid[None, :]) ** 2).sum(axis=1)
    test_preds  = (test_scores > best_thresh).astype(int)
    test_auc    = roc_auc_score(y_test, test_scores)
    print(f"\n=== {DATASET_CHOICE} Test Results ===")
    print(f"AUC: {test_auc:.4f}")
    print(classification_report(y_test, test_preds, target_names=['Normal', 'Anomaly']))

    # ── Save artifacts to Drive ────────────────────────────────────────────────
    import torch as _torch
    _torch.save(final_model.state_dict(), f'{EXPORT_DIR}/encoder_{DATASET_CHOICE}.pt')
    _joblib.dump(scaler, f'{EXPORT_DIR}/scaler_{DATASET_CHOICE}.joblib')
    np.save(f'{EXPORT_DIR}/centroid_{DATASET_CHOICE}.npy', final_centroid)
    # Save held-out test split so notebook 02 evaluates on the exact same data
    # that was never seen during training, validation, or threshold calibration.
    np.save(f'{EXPORT_DIR}/X_test_{DATASET_CHOICE}.npy', X_test_scaled)
    np.save(f'{EXPORT_DIR}/y_test_{DATASET_CHOICE}.npy', y_test)
    meta = {
        'dataset':    DATASET_CHOICE,
        'threshold':  float(best_thresh),
        'val_auc':    float(final_auc),
        'test_auc':   float(test_auc),
        'best_config': best_config,
        'feature_dim': FEATURE_DIM,
        'embed_dim':   best_config['embed_dim'],
        'proj_dim':    best_config['proj_dim'],
    }
    with open(f'{EXPORT_DIR}/meta_{DATASET_CHOICE}.json', 'w') as mf:
        _json.dump(meta, mf, indent=2)
    print(f"Artifacts saved: encoder_{DATASET_CHOICE}.pt | scaler | centroid | meta")

    # ── Training curve plot ────────────────────────────────────────────────────
    import matplotlib.pyplot as plt
    epochs_plot  = [h['epoch'] for h in final_history if 'val_auc' in h]
    aucs_plot    = [h['val_auc'] for h in final_history if 'val_auc' in h]
    train_losses = [h['train_loss'] for h in final_history]
    epochs_all   = [h['epoch'] for h in final_history]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(epochs_all, train_losses, label='Train Loss', color='steelblue')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('NT-Xent Loss')
    axes[0].set_title(f'{DATASET_CHOICE} — Training Loss'); axes[0].legend()
    axes[1].plot(epochs_plot, aucs_plot, marker='o', color='darkorange', label='Val AUC')
    axes[1].axhline(test_auc, color='red', linestyle='--', label=f'Test AUC={test_auc:.4f}')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('ROC-AUC')
    axes[1].set_title(f'{DATASET_CHOICE} — Validation AUC'); axes[1].legend()
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/training_curves.png', dpi=150)
    plt.show()

    TRAINED_MODELS[DATASET_CHOICE] = {
        'model':     final_model,
        'centroid':  final_centroid,
        'scaler':    scaler,
        'threshold': best_thresh,
        'val_auc':   final_auc,
        'test_auc':  test_auc,
    }

# ── Summary ────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("ALL THREE MODELS TRAINED")
print("="*60)
for ds, m in TRAINED_MODELS.items():
    print(f"  {ds}: val_auc={m['val_auc']:.4f}  test_auc={m['test_auc']:.4f}")

best_ds = max(TRAINED_MODELS, key=lambda ds: TRAINED_MODELS[ds]['val_auc'])
print(f"\nBest model: {best_ds} (val_auc={TRAINED_MODELS[best_ds]['val_auc']:.4f})")
print("Run 02_et_ssl_eval_export_clean.ipynb to evaluate the best model on all datasets.")
